# Chapter 4 Project — Spam Classifier
**Comparing All 6 Classification Algorithms**

---

## The Problem

Suresh's Kerala startup is drowning in spam. You've learned 6 classification algorithms across Chapter 4. Now the real question:

> *Which algorithm should actually go into production — and why?*

This notebook runs all 6 algorithms on the same SMS Spam dataset and compares them honestly — not just accuracy, but F1, AUC, training time, and prediction time.

**Algorithms:**
1. Logistic Regression
2. K-Nearest Neighbours (KNN)
3. Decision Tree
4. Random Forest
5. Support Vector Machine (SVM)
6. Naive Bayes

---

## What We're Measuring

| Metric | Why it matters for spam |
|--------|-------------------------|
| **Precision** | Of emails flagged as spam, how many actually were? (false alarms = lost legitimate mail) |
| **Recall** | Of actual spam, how many did we catch? (missed spam = inbox pollution) |
| **F1** | Balance of precision and recall |
| **AUC** | How well the model ranks spam vs ham across ALL thresholds |
| **Train time** | Can it scale to millions of emails? |
| **Predict time** | Can it classify in real-time as emails arrive? |

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import warnings
warnings.filterwarnings('ignore')

# Algorithms
from sklearn.linear_model    import LogisticRegression
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.naive_bayes     import MultinomialNB

# Text processing
from sklearn.feature_extraction.text import TfidfVectorizer

# Utilities
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics         import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score, accuracy_score
)

print("All libraries loaded!")

## Part 1 — Load & Prepare Data

In [ ]:
# Load SMS Spam Collection dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print(f"Dataset: {df.shape[0]} messages")
print(df['label'].value_counts())

# Target: 1 = spam, 0 = ham
df['target'] = (df['label'] == 'spam').astype(int)

X = df['message']
y = df['target']

# Train/test split — stratify keeps spam ratio the same in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")
print(f"Spam in train: {y_train.mean()*100:.1f}% | Spam in test: {y_test.mean()*100:.1f}%")

In [ ]:
# --- WHY TF-IDF instead of CountVectorizer? ---
# CountVectorizer: raw word counts
# TF-IDF: word counts weighted DOWN if the word appears in many messages
#
# Example: the word 'the' appears in almost every message.
# Raw count: 'the' looks important because it's frequent.
# TF-IDF: 'the' gets a low score because it appears everywhere — not informative.
# But 'prize' only appears in spam → gets a HIGH TF-IDF score.
#
# TF-IDF = Term Frequency × Inverse Document Frequency
# Better signal-to-noise ratio than raw counts.

# We fit TfidfVectorizer on TRAIN only — then transform both train and test
# WHY: vocabulary and IDF weights must be learned from training data only
tfidf = TfidfVectorizer(
    stop_words='english',  # remove 'the', 'is', 'at' etc
    max_features=5000,     # top 5000 most informative words
    lowercase=True
)

X_train_tfidf = tfidf.fit_transform(X_train)  # fit + transform on train
X_test_tfidf  = tfidf.transform(X_test)        # transform only on test (no fit!)

print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"Each message → vector of {X_train_tfidf.shape[1]} TF-IDF scores")

## Part 2 — Train All 6 Algorithms

We train each algorithm on the same TF-IDF features and measure training time.

In [ ]:
# Define all 6 algorithms
# Each is configured with reasonable defaults — no heavy tuning yet
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,     # enough iterations to converge
        random_state=42
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5,     # standard starting point
        metric='euclidean' # distance in TF-IDF space
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10,      # limit depth to prevent overfitting
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,  # 100 trees — good balance of speed and accuracy
        random_state=42,
        n_jobs=-1          # use all CPU cores
    ),
    'SVM': SVC(
        kernel='linear',   # linear kernel works well for text (already high-dim)
        probability=True,  # needed for predict_proba → needed for AUC
        random_state=42
    ),
    'Naive Bayes': MultinomialNB(
        alpha=1.0          # Laplace smoothing
        # Note: MultinomialNB needs non-negative features — TF-IDF scores are always ≥ 0 ✓
    )
}

# Train each model and record time
results = {}
trained_models = {}

print(f"{'Model':<22} {'Train Time':>12} {'Predict Time':>14}")
print("-" * 50)

for name, model in models.items():
    # Training time
    t0 = time.time()
    model.fit(X_train_tfidf, y_train)
    train_time = time.time() - t0

    # Prediction time
    t1 = time.time()
    y_pred = model.predict(X_test_tfidf)
    predict_time = time.time() - t1

    y_prob = model.predict_proba(X_test_tfidf)[:, 1]

    results[name] = {
        'Accuracy':     accuracy_score(y_test, y_pred),
        'Precision':    precision_score(y_test, y_pred),
        'Recall':       recall_score(y_test, y_pred),
        'F1':           f1_score(y_test, y_pred),
        'AUC':          roc_auc_score(y_test, y_prob),
        'Train Time':   train_time,
        'Predict Time': predict_time,
        'y_pred':       y_pred,
        'y_prob':       y_prob
    }
    trained_models[name] = model
    print(f"{name:<22} {train_time:>10.3f}s {predict_time:>12.4f}s")

print("\nAll models trained!")

## Part 3 — Results Comparison

In [ ]:
# Build a clean summary table
summary = pd.DataFrame({
    name: {
        'Accuracy':  f"{r['Accuracy']:.4f}",
        'Precision': f"{r['Precision']:.4f}",
        'Recall':    f"{r['Recall']:.4f}",
        'F1':        f"{r['F1']:.4f}",
        'AUC':       f"{r['AUC']:.4f}",
        'Train (s)': f"{r['Train Time']:.3f}",
    }
    for name, r in results.items()
}).T

print("=" * 75)
print("FULL RESULTS — All 6 Algorithms on SMS Spam")
print("=" * 75)
print(summary.to_string())
print()

# Rank by F1
f1_scores = {name: results[name]['F1'] for name in results}
ranked = sorted(f1_scores.items(), key=lambda x: x[1], reverse=True)
print("Ranking by F1 Score:")
for rank, (name, score) in enumerate(ranked, 1):
    print(f"  {rank}. {name:<22} F1 = {score:.4f}")

In [ ]:
# --- VISUALISE: METRIC COMPARISON BAR CHART ---
metrics_to_plot = ['Precision', 'Recall', 'F1', 'AUC']
model_names = list(results.keys())
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Chapter 4 Project — All 6 Algorithms Compared', fontsize=15, fontweight='bold')
axes = axes.flatten()

for ax, metric in zip(axes, metrics_to_plot):
    values = [results[name][metric] for name in model_names]
    bars = ax.bar(model_names, values, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- VISUALISE: ALL ROC CURVES ON ONE PLOT ---
# ROC curves let you compare models across ALL possible thresholds at once

fig, ax = plt.subplots(figsize=(9, 7))

for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={r['AUC']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All 6 Algorithms', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- VISUALISE: ALL CONFUSION MATRICES ---
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Confusion Matrices — All 6 Algorithms', fontsize=14, fontweight='bold')
axes = axes.flatten()

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Ham', 'Spam'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{name}\nF1={r['F1']:.3f} | AUC={r['AUC']:.3f}", fontsize=10)

plt.tight_layout()
plt.show()

print("Reading the confusion matrix:")
print("  Top-left:     True Ham  (correctly identified as ham)")
print("  Top-right:    False Spam (ham flagged as spam — user loses a legit email)")
print("  Bottom-left:  False Ham  (spam slipped through — inbox pollution)")
print("  Bottom-right: True Spam  (correctly caught spam)")

In [ ]:
# --- TRAINING TIME COMPARISON ---
# Speed matters in production — Suresh's server classifies thousands of emails/minute

fig, ax = plt.subplots(figsize=(10, 4))

train_times = [results[name]['Train Time'] for name in model_names]
bars = ax.bar(model_names, train_times, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Training Time Comparison (seconds)', fontweight='bold')
ax.set_ylabel('Time (seconds)')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, train_times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.3f}s', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## Part 4 — Cross Validation

Test set results can get lucky. Cross-validation gives a more honest picture — the model is tested on 5 different splits of the data.

In [ ]:
# 5-fold stratified cross validation on all models
# WHY stratified? Keeps the spam/ham ratio consistent in every fold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print(f"{'Model':<22} {'CV F1 Mean':>12} {'CV F1 Std':>12}")
print("-" * 48)

for name, model in models.items():
    scores = cross_val_score(
        model, X_train_tfidf, y_train,
        cv=cv, scoring='f1', n_jobs=-1
    )
    cv_results[name] = scores
    print(f"{name:<22} {scores.mean():>12.4f} {scores.std():>12.4f}")

print("\nLow std = stable model. High std = results vary depending on which data it sees.")

In [ ]:
# Visualise CV results with error bars (shows stability)
fig, ax = plt.subplots(figsize=(10, 5))

means = [cv_results[name].mean() for name in model_names]
stds  = [cv_results[name].std()  for name in model_names]

bars = ax.bar(model_names, means, yerr=stds, color=colors,
              edgecolor='black', linewidth=0.5, capsize=6, alpha=0.85)
ax.set_title('5-Fold Cross Validation F1 Scores (with std dev)', fontweight='bold')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.1)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)

for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.02,
            f'{mean:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## Part 5 — Final Verdict

Now we make the engineering decision: which algorithm goes into production?

In [ ]:
# Final verdict — updated based on actual experimental results

print("=" * 70)
print("FINAL VERDICT — Which Algorithm for Production Spam Filter?")
print("=" * 70)

print("""
Based on actual 5-fold cross validation results:

🥇 WINNER: SVM (F1 = 0.924)
  → Best performance on high-dimensional TF-IDF features
  → Linear kernel excels when data is already well-separated in high dims
  → Small error bars = stable and reliable across all folds

🥈 STRONG RUNNER-UP: Random Forest (F1 = 0.908)
  → Excellent performance despite not being designed for text
  → Robust to noise — 100 trees voting cancels out individual mistakes
  → Slower to train but very reliable

🥉 SPEED CHAMPION: Naive Bayes (F1 = 0.896)
  → Only 0.028 behind Random Forest but trains in milliseconds
  → Best choice if you need to retrain daily on new spam patterns
  → Gmail's original spam filter was Naive Bayes — this result shows why

4th: Logistic Regression (F1 = 0.783)
  → Underperformed here due to no hyperparameter tuning
  → With C tuning it would likely reach 0.90+
  → Still the best FIRST model to try on any new problem

5th: Decision Tree (F1 = 0.775)
  → Overfits — some spam words dominate the whole tree
  → Unstable — small data change = very different tree

6th WORST: KNN (F1 = 0.394)
  → Curse of dimensionality — 5000 TF-IDF dimensions kills distance metrics
  → Prediction is slow — compares every new email to ALL training emails
  → Exactly as predicted before running experiments
""")

print("KEY INSIGHT:")
print("  The best algorithm depends on your constraint.")
print("  If you want maximum accuracy  → SVM")
print("  If you want maximum speed     → Naive Bayes")
print("  If you want balance           → Random Forest")
print("  Logistic Regression always tried first — tune it before dismissing it")
print("  There is no single best algorithm — only best for a given situation.")


In [ ]:
# --- TEST THE BEST MODEL ON CUSTOM MESSAGES ---
# Find the model with highest F1
best_model_name = max(results, key=lambda x: results[x]['F1'])
best_model = trained_models[best_model_name]

print(f"Best model by F1: {best_model_name}\n")

test_messages = [
    "Congratulations! You have won Rs 1,00,000. Call now to claim your prize!",
    "Hey, are you joining the team lunch at 1pm today?",
    "URGENT: Your SBI account will be blocked. Update KYC immediately.",
    "Please review the Q3 report before tomorrow's meeting.",
    "FREE entry in our lucky draw! Win an iPhone 15. Reply WIN now.",
    "Amma called, she wants you to call back when free."
]

test_tfidf = tfidf.transform(test_messages)  # use the SAME tfidf fitted on training data
preds = best_model.predict(test_tfidf)
probs = best_model.predict_proba(test_tfidf)[:, 1]

print(f"{'Message':<55} {'Result':<8} {'Spam Prob'}")
print("-" * 80)
for msg, pred, prob in zip(test_messages, preds, probs):
    label = "SPAM ⚠️" if pred == 1 else "HAM  ✓"
    print(f"{msg[:53]:<55} {label:<8} {prob:.4f}")

## Summary — Chapter 4 Complete

| Algorithm | Strengths | Weaknesses | Best for |
|-----------|-----------|------------|----------|
| **Logistic Regression** | Fast, interpretable, great for text | Assumes linear boundary | Text classification, baselines |
| **KNN** | Simple, no training | Slow prediction, fails in high dimensions | Small datasets, low dimensions |
| **Decision Tree** | Interpretable, visual | Overfits, unstable | Explainability needed |
| **Random Forest** | Robust, handles noise | Slow, memory heavy | Tabular data, general purpose |
| **SVM** | Excellent in high dimensions | Slow on huge data, needs tuning | Text, images, small-medium data |
| **Naive Bayes** | Extremely fast, great for text | Assumes independence | Text, real-time systems, email |

---

## The Core Lesson of Chapter 4

> There is no universally best algorithm. The right algorithm depends on your data type, dimensionality, dataset size, speed requirements, and interpretability needs.
> 
> An ML engineer's job is not to memorise which algorithm is "best" — it's to understand WHY each algorithm behaves the way it does, and match that to the problem.

---

## Practice Task

You've seen spam classification. Now try a different text problem.

The [20 Newsgroups dataset](https://scikit-learn.org/stable/datasets/real_world.html#newsgroups-dataset) contains news articles labelled by topic (sports, politics, technology, etc.).

**Task:** Load it using `sklearn.datasets.fetch_20newsgroups`, pick any 3 categories, and compare Logistic Regression vs Naive Bayes vs Random Forest on **multi-class** classification (not binary). Which wins?

In [ ]:
# YOUR CODE HERE
# Hint:
# from sklearn.datasets import fetch_20newsgroups
# categories = ['sci.space', 'rec.sport.cricket', 'talk.politics.misc']
# data = fetch_20newsgroups(subset='train', categories=categories)
# X, y = data.data, data.target